# 02 · The verifier and the MATH-500 baseline

Companion notebook for [Chapter 3](../chapters/ch03-verifier.md).

**You will:** run the four-stage verifier, build a test table of your own, inspect
the dataset, and measure the base model's baseline.

## 0 · Setup

Run this cell first. It picks a device and imports everything the notebook needs.

If you get `ModuleNotFoundError: reasoning_from_scratch`, your kernel is not the
course environment — see the [setup page](../start/setup.md).

In [ ]:
import torch
from reasoning_from_scratch.ch02 import get_device

device = get_device()
print("torch", torch.__version__)

## 1 · Extraction

The convention: the final answer goes in `\boxed{}`. The extractor takes the
**last** one and counts braces so nested fractions survive.

In [ ]:
from reasoning_from_scratch.ch03 import extract_final_candidate

cases = [
    r"first \boxed{12} then \boxed{\frac{3}{4}}",   # nested braces, takes the last
    r"\boxed{83}",
    "The final answer is 83",                        # no box: falls back to last number
    "no answer here at all",
]
for c in cases:
    print(f"{c!r:55} -> {extract_final_candidate(c)!r}")

print()
print("strict mode (reward):", repr(extract_final_candidate("The final answer is 83", fallback=None)))

**Note the difference.** Evaluation uses `fallback="number_then_full"` and is
lenient. The RL reward in chapter 6 uses `fallback=None` and is strict — the model
must learn the format.

## 2 · Grading: symbolic equivalence

In [ ]:
from reasoning_from_scratch.ch03 import grade_answer

tests = [
    (r"\frac{1}{2}", "0.5",      True,  "symbolic equivalence"),
    (r"50\%",        "50",       True,  "percent stripped"),
    (r"\sqrt{49}",   "7",        True,  "sqrt evaluates"),
    (r"45^\circ",    "45",       True,  "degree marker stripped"),
    ("2^2",          "4",        True,  "power"),
    ("2x",           "2*x",      True,  "implicit multiplication"),
    ("38",           "83",       False, "genuinely different"),
    ("0.333",        r"\frac{1}{3}", False, "strict: no tolerance"),
]
ok = 0
for pred, gt, expect, why in tests:
    got = grade_answer(pred, gt)
    ok += (got == expect)
    flag = "ok " if got == expect else "!! "
    print(f"{flag}{pred!r:18} vs {gt!r:12} -> {got!s:5} (expect {expect})  {why}")
print(f"\n{ok}/{len(tests)} as expected")

## 3 · Write your own tests

**Before running**, predict each result. Add at least five cases you think are
tricky: units, intervals, `x = 3`, a percentage of a percentage, a 3,000-character
string.

In [ ]:
my_tests = [
    # (prediction, ground_truth, your_expected_result)
    ("3", "3.0", True),
    ("\\(3, 4\\)", "(3, 4)", True),
    ("(3, \\frac{\\pi}{2})", "(3, pi/2)", None),   # TRY: what do you predict?
    ("1/0", "inf", None),
    ("x=3", "3", None),
]
for pred, gt, expect in my_tests:
    print(f"{pred!r:30} vs {gt!r:10} -> {grade_answer(pred, gt)}   (you expected {expect})")

**The third row is a real false negative.** It returns `False`: the left side
normalises to `3, (\pi)/(2)`, which no longer starts with a bracket, so
`split_into_parts` returns one part instead of two. This is exactly the kind of
quiet error that caps your measured accuracy — see
`ch03/03_advanced-parser` in the original repo.

## 4 · The dataset

In [ ]:
from reasoning_from_scratch.ch03 import load_math500_test, render_prompt

math500 = load_math500_test()
print("items:", len(math500))
print("fields:", list(math500[0].keys()))

item = math500[0]
print("\nsubject:", item["subject"], "| level:", item["level"])
print("problem:", item["problem"][:200])
print("answer :", item["answer"])
print("\n--- rendered prompt ---")
print(render_prompt(item["problem"]))

## 5 · Baseline accuracy

Full 500 questions takes 10 minutes on a strong GPU and ~40 on a laptop CPU. Start
with 10 to confirm the pipeline, then scale.

**Expected:** 15.6% (78/500) on the full set. On 10 questions you will see anything
from 10% to 40% — the subset is too small to be precise.

In [ ]:
from reasoning_from_scratch.ch03 import load_model_and_tokenizer, evaluate_math500_stream

model, tokenizer = load_model_and_tokenizer(
    which_model="base", device=device, use_compile=False
)

records = evaluate_math500_stream(
    model=model, tokenizer=tokenizer, device=device,
    math_data=math500[:10],            # TRY: 50, then 500
    max_new_tokens=512,
    verbose=False,
)

## 6 · Inspect what the model actually said

Accuracy is a number; the transcripts tell you *why*. Look at a wrong answer.

In [ ]:
import json

# the evaluator writes a JSONL next to your working directory
import glob
for path in glob.glob("math500*.jsonl"):
    print("log:", path)
    with open(path) as f:
        rows = [json.loads(line) for line in f]
    wrong = [r for r in rows if not r.get("correct", False)]
    print(f"{len(wrong)} wrong of {len(rows)}")
    if wrong:
        print("\n--- one wrong answer ---")
        print(str(wrong[0].get("generated_text", ""))[:1200])
    break

**What to look for:** did it answer correctly but forget the box? Did it box the
wrong thing? Did it run out of tokens? Each is a different bug, and only the last
one is a compute problem.

## 7 · Exercises

1. Run `--dataset_size 50` via the CLI script and compare against your in-notebook
   number. Same ballpark?
2. Grade 20 wrong answers by hand. How many were actually right? That is your
   verifier's false-negative rate.
3. What does `grade_answer` do with a 3,000-character input, and why?